In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType
)
from datetime import datetime
world_state_schema = StructType([
    StructField("store",               StringType()),
    StructField("inventory_gallons",   IntegerType()),
    StructField("weather_temp_f",      IntegerType()),
    StructField("weather_condition",   StringType()),
    StructField("event_timestamp",     TimestampType()),
    StructField("source",              StringType())
])
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.supply_agent_demo")
spark.sql("DROP TABLE IF EXISTS workspace.supply_agent_demo.world_state")
print("Schema ready. world_state dropped for clean run.")
## Version 0: Baseline Readings
Normal conditions from 07:00 to 08:00. Inventory is healthy (110-120 gallons). Weather is rainy and cool (45-46F). This is what a normal morning looks like before anything goes wrong.
baseline_readings = [
    ("Downtown", 120, 45, "Rainy",  datetime(2026, 5, 19, 7,  0), "iot_sensor"),
    ("Downtown", 115, 45, "Rainy",  datetime(2026, 5, 19, 7, 30), "iot_sensor"),
    ("Downtown", 110, 46, "Cloudy", datetime(2026, 5, 19, 8,  0), "iot_sensor"),
]

df_baseline = spark.createDataFrame(baseline_readings, world_state_schema)

df_baseline.write \
    .format("delta") \
    .option("userMetadata", "baseline_normal") \
    .mode("overwrite") \
    .saveAsTable("workspace.supply_agent_demo.world_state")

print("Baseline written. Delta version: 0")
print("3 normal readings committed: 07:00, 07:30, 08:00")
## Version 1: Corrupted Readings at 08:12
A faulty IoT sensor fires and reports inventory as -999 gallons. Simultaneously a weather API glitch pushes a 102F heatwave warning. This is the reading the agent will see at 08:14 when it runs its decision cycle.
corrupted_readings = [
    ("Downtown", -999, 102, "Extreme Heatwave",
     datetime(2026, 5, 19, 8, 12), "iot_sensor_faulty"),
]

df_corrupted = spark.createDataFrame(corrupted_readings, world_state_schema)

df_corrupted.write \
    .format("delta") \
    .option("userMetadata", "corrupted_sensor") \
    .mode("append") \
    .saveAsTable("workspace.supply_agent_demo.world_state")

print("Corrupted readings injected. Delta version: 1")
print("Inventory: -999 gal | Temp: 102F | Source: iot_sensor_faulty")
## Version 2: Corrected Readings at 08:16
The APIs resolve the glitch four minutes later and push accurate values. A new correct row is appended. The live table now returns this row as the latest. The corrupted window existed for only 4 minutes but the agent already acted on it.
corrected_readings = [
    ("Downtown", 118, 45, "Rainy",
     datetime(2026, 5, 19, 8, 16), "iot_sensor"),
]

df_corrected = spark.createDataFrame(corrected_readings, world_state_schema)

df_corrected.write \
    .format("delta") \
    .option("userMetadata", "self_corrected") \
    .mode("append") \
    .saveAsTable("workspace.supply_agent_demo.world_state")

print("Corrected readings written. Delta version: 2")
print("Inventory: 118 gal | Temp: 45F | Source: iot_sensor")
## What the Live Table Shows Right Now
This is what any analyst sees if they query the table today. The -999 reading is present in the append log but the latest reading looks normal. Without Delta Time Travel, the corrupted window is invisible.
print("Live table, all rows ordered by event_timestamp:")
spark.sql("""
    SELECT store, inventory_gallons, weather_temp_f,
           weather_condition, event_timestamp, source
    FROM workspace.supply_agent_demo.world_state
    WHERE store = 'Downtown'
    ORDER BY event_timestamp DESC
""").show()
## Delta History: Three Immutable Versions
Every write is permanently versioned. The corrupted_sensor commit at version 1 cannot be deleted, only superseded. This is what makes Time Travel possible.
print("Delta commit history:")
spark.sql("""
    DESCRIBE HISTORY workspace.supply_agent_demo.world_state
""").select("version", "timestamp", "operation", "userMetadata").show(truncate=False)
## Sensor Timeline Visualization
The chart below shows inventory levels and temperature over the 90-minute window. The corrupted reading at 08:12 is immediately visible as a sharp anomaly.
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

df_timeline = spark.sql("""
    SELECT event_timestamp, inventory_gallons, weather_temp_f,
           weather_condition, source
    FROM workspace.supply_agent_demo.world_state
    WHERE store = 'Downtown'
    ORDER BY event_timestamp ASC
""").toPandas()

df_timeline["event_timestamp"] = pd.to_datetime(df_timeline["event_timestamp"])
df_timeline["time_label"] = df_timeline["event_timestamp"].dt.strftime("%H:%M")
is_faulty = df_timeline["source"] == "iot_sensor_faulty"

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.suptitle(
    "Downtown Seattle Store - Sensor Readings: 07:00 to 08:20\n"
    "Corrupted window at 08:12 is invisible in the live table without Time Travel",
    fontsize=13, fontweight="bold", y=0.98
)

bar_colors = ["#d62728" if faulty else "#1f77b4" for faulty in is_faulty]
ax1.bar(df_timeline["time_label"], df_timeline["inventory_gallons"],
        color=bar_colors, width=0.5, edgecolor="white")
ax1.axhline(y=0, color="#d62728", linestyle="--", linewidth=1, alpha=0.6)
ax1.axhline(y=50, color="#ff7f0e", linestyle="--", linewidth=1, alpha=0.6)
ax1.set_ylabel("Inventory (gallons)", fontsize=11)
ax1.set_title("Oat Milk Inventory", fontsize=11)
ax1.set_ylim(-1100, 200)
normal_patch = mpatches.Patch(color="#1f77b4", label="Normal reading")
faulty_patch = mpatches.Patch(color="#d62728", label="Corrupted reading")
ax1.legend(handles=[normal_patch, faulty_patch], fontsize=9, loc="lower right")
for i, (val, faulty) in enumerate(zip(df_timeline["inventory_gallons"], is_faulty)):
    ax1.text(i, val + 5 if val >= 0 else val - 60, str(val),
             ha="center", va="bottom", fontsize=9,
             color="#d62728" if faulty else "#1f77b4", fontweight="bold")

ax2.plot(df_timeline["time_label"], df_timeline["weather_temp_f"],
         color="#2ca02c", linewidth=2, marker="o", markersize=6)
ax2.scatter(df_timeline.loc[is_faulty, "time_label"],
            df_timeline.loc[is_faulty, "weather_temp_f"],
            color="#d62728", zorder=5, s=100, label="Corrupted reading")
ax2.axhline(y=90, color="#d62728", linestyle="--", linewidth=1, alpha=0.6,
            label="Emergency threshold (90F)")
ax2.set_ylabel("Temperature (F)", fontsize=11)
ax2.set_xlabel("Time", fontsize=11)
ax2.set_title("Weather Temperature", fontsize=11)
ax2.legend(fontsize=9, loc="upper right")
for i, (val, faulty) in enumerate(zip(df_timeline["weather_temp_f"], is_faulty)):
    ax2.text(i, val + 1.5, f"{val}F", ha="center", va="bottom", fontsize=9,
             color="#d62728" if faulty else "#2ca02c", fontweight="bold")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
